# Sightings Follow the Boat Timetable

If the public record is a record of boat trips, sightings should bunch up in the hours the boats are
at sea, not spread across daylight the way whales are. Monterey Bay Whale Watch runs a morning trip
from 9:00 (3.5 to 4 hours) and an afternoon trip from 13:30 (3 hours)
([schedule](https://montereybaywhalewatch.com/schedule/), read 16 October 2026).

`local_hour` is the clock hour written on each record. Only about one timed record in ten states a
time zone (`has_timezone`), nearly always a Pacific offset, so hours are read as local time and the
share for zone-stating records alone is reported beside the main figure.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## Timed Public Sightings in Each Hour

In [ ]:
%%sql
SELECT
    fct_sighting.local_hour,
    COUNT(*) AS public_records,
    COUNT_IF(fct_sighting.has_timezone) AS zone_known_records
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
  AND fct_sighting.local_hour IS NOT NULL
GROUP BY fct_sighting.local_hour
ORDER BY fct_sighting.local_hour

## The Shares
Every share is of sightings logged in daylight, 6:00 to 19:59 (`is_daylight_hour`).
* **10:00 to 11:59** (`is_peak_hour`): the two peak hours, for all daylight sightings and for those that
  state a time zone. Spread evenly over the 14 daylight hours, two hours would hold 14%.
* **While a trip is at sea** (`trip_at_sea_weight`): 9:00 to 13:00 and 13:30 to 16:30. A trip is out for
  half of hours 13 and 16, so a sighting in those hours counts half. The 7 hours are half of daylight, so
  an even spread would put 50% of sightings in them.

In [ ]:
%%sql
SELECT
    COUNT(*) AS daylight_records,
    ROUND(COUNT_IF(fct_sighting.is_peak_hour) / daylight_records, 3) AS share_10_to_12,
    ROUND(SUM(fct_sighting.trip_at_sea_weight) / daylight_records, 3) AS share_while_trips_at_sea,
    COUNT_IF(fct_sighting.has_timezone) AS zone_known_records,
    ROUND(
        COUNT_IF(fct_sighting.is_peak_hour AND fct_sighting.has_timezone) / zone_known_records,
        3
    ) AS share_10_to_12_zone_known
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
  AND fct_sighting.is_daylight_hour

## Result
Of public sightings logged in daylight, **44%** fall in two hours, 10:00 to 11:59, against 14% if spread
evenly, and 89% while a scheduled trip is at sea, against 50%. Records that state a time zone give 38% for
the two peak hours.

## Limits
* People are out most at mid-morning anyway, so the two peak hours alone overstate the timetable; the
  share while trips are at sea is the firmer measure.
* The trip times are one operator's current schedule, applied to twenty years and three harbors.
* Nine timed records in ten have no time zone.